In [2]:
"""
Imports variables from urban_data_variables module.
Replaces 'obns_lat' from district_codes with the correct latin spelling.
"""

import pandas as pd
import geopandas as gpd
import re
from urban_data_variables import (
    districts,
    district_codes,
    schools,
    kindergartens,
    parking_spaces,
    public_transport_accessibility,
    parks_accessibility,
    healthcare_coverage
)
pd.set_option('display.max_rows', None)


In [3]:
import cyrtranslit

# Applies a lambda function to latinize district names
district_codes['obns_lat'] = district_codes['obns_cyr'].apply(lambda x: cyrtranslit.to_latin(x, 'bg'))
# Removes diacritical A
district_codes['obns_lat'] = district_codes['obns_lat'].apply(lambda x: x.replace('Ă', 'A'))
district_codes


,id,obns_num,obns_cyr,obns_lat,geometry
0,3,09,ЛОЗЕНЕЦ,LOZENETS,"MULTIPOLYGON (((23.3095 42.67248, 23.31027 42...."
1,4,10,ТРИАДИЦА,TRIADITSA,"MULTIPOLYGON (((23.28299 42.65376, 23.28319 42..."
2,5,17,ВИТОША,VITOSHA,"MULTIPOLYGON (((23.13931 42.59156, 23.13938 42..."
3,6,23,ПАНЧАРЕВО,PANCHAREVO,"MULTIPOLYGON (((23.28519 42.55851, 23.2853 42...."
4,7,18,ОВЧА КУПЕЛ,OVCHA KUPEL,"MULTIPOLYGON (((23.13952 42.6773, 23.13957 42...."
5,8,11,КРАСНА ПОЛЯНА,KRASNA POLYANA,"MULTIPOLYGON (((23.23027 42.69073, 23.23028 42..."
6,9,05,СЕРДИКА,SERDIKA,"MULTIPOLYGON (((23.29775 42.71481, 23.29794 42..."
7,11,13,НАДЕЖДА,NADEZHDA,"MULTIPOLYGON (((23.27909 42.76763, 23.27959 42..."
8,12,21,НОВИ ИСКЪР,NOVI ISKAR,"MULTIPOLYGON (((23.24235 42.81768, 23.24238 42..."
9,13,20,ВРЪБНИЦА,VRABNITSA,"MULTIPOLYGON (((23.20036 42.7711, 23.20041 42...."


In [4]:
neighborhoods_list = districts['regname'].tolist()
neighborhoods_list = [cyrtranslit.to_latin(n, 'bg') for n in neighborhoods_list]
neighborhoods_list = [n.replace('Ă', 'A').replace('ă', 'a') for n in neighborhoods_list]

# Split several names into different entries
neighborhoods_list = sorted({part.strip() for n in neighborhoods_list for part in re.split(r',', n) if part.strip()})

# Remove small words
neighborhoods_list = [re.sub(r'\b\w{1,2}\.', '', n) for n in neighborhoods_list]
neighborhoods_list = [n.strip() for n in neighborhoods_list]

def tidy(n):
    n = re.sub(r'\s*-\s*(\d+)\s*mr\b', r' \1', n, flags=re.IGNORECASE)  # "Lyulin - 10 mr" -> "Lyulin 10"
    return re.sub(r'\s+', ' ', n).strip()

neighborhoods_list = sorted({tidy(n) for n in neighborhoods_list})

neighborhoods_list

['"SHTetnitsa"',
 '1',
 '3-ta izvangradska chast',
 'Abdovitsa',
 'Avramitsa"',
 'BAN 4km',
 'Balsha',
 'Balsha - razshirenie',
 'Balsha poleto',
 'Bankya',
 'Bankya - Verdikal',
 'Bankya - razshirenie',
 'Bardo - Dimitar',
 'Barite',
 'Batareya',
 'Bel Kamak',
 'Beli Breg - Grobishtata',
 'Beli Brezi',
 'Beli rid',
 'Belikata',
 'Belovodski pat',
 'Benkovski',
 'Benkovski - razshirenie',
 'Berilitsa',
 'Bistritsa',
 'Bistritsa - Kalfin dol',
 'Bistritsa - Kokalyane - pole',
 'Bistritsa - SHiranata',
 'Bistritsa - TSonkina mahala',
 'Bistritsa - razsh. TSonkina mahala',
 'Bistritsa - razsh. iztok',
 'Bistritsa - rezervata',
 'Bitpazar Malashevtsi',
 'Bivolyarnik',
 'Bivshi kazarmi GUSV Smardan',
 'Biznes zona Progono',
 'Biznes zona bul. Iztochna tangenta - Vrazhdebna',
 'Blatata Krivina',
 'Bogrov',
 'Bogrovski pesotsi',
 'Borisova gradina - Loven park',
 'Borisova gradina - Pogrebite chast 1',
 'Borisova gradina - Seminariyata',
 'Borisova gradina - sever',
 'Borisova gradina - yug',

In [5]:
"""
Mapping districts and neighborhoods to the scraped listings files
"""

sofia_for_rent_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_rent_scraped.csv')

districts_list = district_codes['obns_lat'].tolist()

pattern_districts = '|'.join(re.escape(d) for d in sorted(districts_list, key=len, reverse=True))
pattern_neighborhoods = '|'.join(re.escape(d) for d in sorted(neighborhoods_list, key=len, reverse=True))

sofia_for_rent_scraped['district'] = sofia_for_rent_scraped['location'].str.extract(rf'\b({pattern_districts})\b', flags=re.IGNORECASE)[0]
sofia_for_rent_scraped['neighborhood'] = sofia_for_rent_scraped['location'].str.extract(rf'\b({pattern_neighborhoods})\b', flags=re.IGNORECASE)[0]

# match rate
len(sofia_for_rent_scraped[(sofia_for_rent_scraped['neighborhood'].notna()) | (sofia_for_rent_scraped['district'].notna())]) / len(sofia_for_rent_scraped)

0.5711145096539858

In [6]:
sofia_for_sale_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_sale_scraped.csv')

sofia_for_sale_scraped['district'] = sofia_for_sale_scraped['location'].str.extract(rf'\b({pattern_districts})\b', flags=re.IGNORECASE)[0]
sofia_for_sale_scraped['neighborhood'] = sofia_for_sale_scraped['location'].str.extract(rf'\b({pattern_neighborhoods})\b', flags=re.IGNORECASE)[0]

# match rate
len(sofia_for_sale_scraped[(sofia_for_rent_scraped['neighborhood'].notna()) | (sofia_for_sale_scraped['district'].notna())]) / len(sofia_for_rent_scraped)

0.6298030969221946

In [7]:
to_remove = ['Sofia,', 's.', 'v.z.', 'zh.gr.', 'zh.k.', 'm.', 'kv.', 'm-t']

def clean_neighborhood(name):
    name = cyrtranslit.to_latin(name, 'bg').replace('Ă', 'A').replace('ă', 'a')
    for r in to_remove:
        name = name.replace(r, '')
    name = (name.replace('"', '')
            .replace('Belite', 'Beli')
            .replace('Botunets 2', 'Botunets')
            .replace('3 chast', '')
            .replace('.  Gabaro - Azmata', '')
            .replace('Krasna polyana 1', 'Krasna polyana 1/2')
            .replace('Krasna polyana 2', 'Krasna polyana 1/2')
            .replace('Krasna polyana - chast 3', 'Krasna polyana 3')
            .replace('Krasna polyana - chast 1 i 2', 'Krasna polyana 1/2')
            .replace('Levski V', 'Levski')
            .replace('Ovcha Kupel - 1', 'Ovcha Kupel 1')
            .replace('Ovcha kupel 2', 'Ovcha kupel')
            .replace('Ovcha Kupel - yug', 'Ovcha Kupel')
            .replace('Ovcha Kupel - sever', 'Ovcha Kupel')
            .replace('Ovcha Kupel -  50', 'Ovcha Kupel')
            .replace('Republika 2', 'Republika')
            .replace('Vitosha - VETS Simeonovo', 'Simeonovo')
            .replace('Dragalevtsi - razshirenie - sever', 'Dragalevtsi')
            .replace('Dragalevtsi, v. Miramonte', 'Dragalevtsi')
            .strip())
    name = re.sub(r'^(Simeonovo|Dragalevtsi)$', 'Simeonovo - Dragalevtsi', name)
    name = re.sub(r'^.*Vrana.*$', 'Vrana', name)
    name = re.sub(r'^.*German.*$', 'German', name)
    name = re.sub(r'^(Vrana|German)$', 'Vrana - German', name)
    return re.sub(r'\s*-\s*(\d+)\s*mr\b', r' \1', name).lower()

In [24]:
combined_listings = pd.concat([sofia_for_sale_scraped, sofia_for_rent_scraped], ignore_index=True)

combined_listings['location_clean'] = combined_listings['location'].apply(clean_neighborhood)

combined_listings.sample(5)

,url,ad_id,listing_type,location,price_eur,area_sqm,price_per_sqm_eur,floor,central_heating,gas,...,description_bulgarian,features,views,saves,ad_published_date,agency,date_scraped,district,neighborhood,location_clean
9636,https://imoti.info/en/properties/83852432-for-...,83852432,2 bedroom,"Sofia, Lyulin - tsentar",271 220,135,"2 009,04",3rd,NO,NO,...,Агенция SUPRIMMO: www.suprimmo.bg\nПредставяме...,"[""Lift""]",1191,1,12 July 2026,SUPRIMMO - GLAVEN OFIS,2026-09-25 18:39:02,Lyulin,Lyulin - tsentar,lyulin - tsentar
34593,https://imoti.info/en/properties/51908257-for-...,51908257,2 bedroom,"Sofia, Tsentar",990,90,11,3rd,LOCAL HEATING,-,...,Ел-Ем предлага стилно обзавден 3-стаен апартам...,"[""Furnished"", ""Security"", ""Air-condition"", ""Ac...",482,1,15 September 2026,AGENTSIYA EL-EM EOOD,2026-09-25 16:26:14,NaN,NaN,tsentar
9369,https://imoti.info/en/properties/95551979-for-...,95551979,2 bedroom,"Sofia, Lozenets",660 000,156,"4 230,77",5th,LOCAL HEATING,-,...,"Топ локация, 600 метра от НДК, до градина 'Бан...","[""Lift"", ""With a garage"", ""With a parking lot""...",891,0,1 September 2026,ESTEYT MARK,2026-09-25 18:36:20,Lozenets,NaN,lozenets
14280,https://imoti.info/en/properties/38555958-for-...,38555958,3 bedroom,"Sofia, Manastirski livadi",391 300,170,"2 301,76",3rd,-,-,...,Предлагаме за продажба просторен и светъл апар...,"[""Lift"", ""With a garage"", ""With a parking lot""]",382,0,8 September 2026,ALDES IMOTI,2026-09-25 19:24:33,NaN,NaN,manastirski livadi
9222,https://imoti.info/en/properties/11552698-for-...,11552698,1 bedroom,"Sofia, Lozenets",415 000,82,"5 060,98",2nd,YES,NaN,...,BULGARIA SOTHEBY’S INTERNATIONAL REALTY има уд...,"[""Lift"", ""Security cameras"", ""Access control"",...",59,0,Обявата е коригирана на 10 September 2026,BALGARIYA SOTBIS INTERNESHANAL RIALTI,2026-09-25 18:34:51,Lozenets,NaN,lozenets


In [9]:
districts['regname_clean'] = districts['regname'].apply(clean_neighborhood)

districts[districts['regname_clean'].str.contains('b-5', case=False)]

,object_id,regname,rajon,area_sos500,rzp_jilsgr,rzp_all,redukcia_20%,ppl_30kvm,ppl_35kvm,ppl_40kvm,area_ge,dens30_ppl_kvkm,dens35_ppl_kvkm,dens40_ppl_kvkm,id,area_kv_m,geometry,regname_clean


In [27]:
regname_clean_set = {s.strip().lower() for s in districts['regname_clean'].dropna()}         
location_clean_set = {s.strip().lower() for s in combined_listings['location_clean'].dropna()} 

matches = {x: [y for y in regname_clean_set if x in y] for x in location_clean_set}

matched = {x: ys for x, ys in matches.items() if ys}
unmatched = {x for x, ys in matches.items() if not ys}

unmatched

{'7-mi 11-ti kilometar',
 'doktorski pametnik',
 'eksperimentalen',
 'oborishte',
 'poligona',
 'reduta',
 'zona b-18',
 'zona b-19',
 'zona b-5',
 'zona b-5-3'}

In [11]:
districts['regname_clean_2'] = districts['regname_clean'].apply(
    lambda r: max([x for x in location_clean_set if x in r], key=len, default=None)
    if isinstance(r, str) else None
)

In [12]:
lookup = (
    districts
    .dropna(subset=['regname_clean_2'])
    .assign(exact=lambda d: d['regname_clean'].str.strip().str.lower() == d['regname_clean_2'])
    .sort_values('exact', ascending=False)
    .drop_duplicates('regname_clean_2')
    .drop(columns='exact')
)

merge = combined_listings.merge(
    lookup, how='left',
    left_on='location_clean', right_on='regname_clean_2',
    validate='m:1'
)

merge.columns

Index(['url', 'ad_id', 'listing_type', 'location', 'price_eur', 'area_sqm',
       'price_per_sqm_eur', 'floor', 'central_heating', 'gas',
       'construction_type', 'construction_year', 'description_bulgarian',
       'features', 'views', 'saves', 'ad_published_date', 'agency',
       'date_scraped', 'district', 'neighborhood', 'location_clean',
       'object_id', 'regname', 'rajon', 'area_sos500', 'rzp_jilsgr', 'rzp_all',
       'redukcia_20%', 'ppl_30kvm', 'ppl_35kvm', 'ppl_40kvm', 'area_ge',
       'dens30_ppl_kvkm', 'dens35_ppl_kvkm', 'dens40_ppl_kvkm', 'id',
       'area_kv_m', 'geometry', 'regname_clean', 'regname_clean_2'],
      dtype='str')

In [43]:
districts[districts['rajon'].str.contains('Ц', case=False)]

,object_id,regname,rajon,area_sos500,rzp_jilsgr,rzp_all,redukcia_20%,ppl_30kvm,ppl_35kvm,ppl_40kvm,area_ge,dens30_ppl_kvkm,dens35_ppl_kvkm,dens40_ppl_kvkm,id,area_kv_m,geometry,regname_clean,regname_clean_2
12,108,Рудник Кремиковци 2,Нови Искър / Кремиковци,NaN,177.0,177,142,5,4,4,6002209,1,1,1,a044af65-423a-4ce7-9fd1-094c343281f2,6.001423e+06,"MULTIPOLYGON (((23.46077 42.79646, 23.4605 42....",rudnik kremikovtsi 2,kremikovtsi
16,53,Южен парк част 3,Триадица,637.0,11181.0,11818,9454,315,270,236,731978,430,369,322,a4daef31-b41d-4c18-9666-4c594e728a0e,7.318795e+05,"MULTIPOLYGON (((23.30648 42.65691, 23.30648 42...",yuzhen park chast 3,yuzhen park
18,135,"ж.к. Надежда 3, ж.к. Връбница 1Б",Надежда / Връбница,267738.0,10824.0,278562,222850,7428,6367,5571,424525,17497,14998,13123,1b768ad0-25d7-4813-ad24-fd8cdc3bfe5f,4.244735e+05,"MULTIPOLYGON (((23.29632 42.73228, 23.29607 42...","nadezhda 3, vrabnitsa 1b",vrabnitsa 1
23,57,кв. Хладилника,Лозенец,262528.0,1682.0,264210,211368,7046,6039,5284,181424,38837,33287,29125,9c3d1e3f-aa1d-42d7-817c-50f5556a23a6,1.813986e+05,"MULTIPOLYGON (((23.31649 42.6593, 23.31375 42....",hladilnika,hladilnika
32,130,кв. Кръстова вада - изток,Лозенец,146761.0,30394.0,177155,141724,4724,4049,3543,589343,8016,6870,6012,1ff6154b-187d-4432-b8a7-1a47c20e357a,5.892638e+05,"MULTIPOLYGON (((23.32148 42.6567, 23.32147 42....",krastova vada - iztok,krastova vada
35,63,ЦГЧ Зона Г8 част Триадица,Триадица,276037.0,4650.0,280687,224550,7485,6416,5614,273390,27378,23468,20535,12c92d74-773d-46b3-af50-46413fb35149,2.733512e+05,"MULTIPOLYGON (((23.31897 42.689, 23.31805 42.6...",tsgch zona g8 chast triaditsa,NaN
38,66,ЦГЧ Зона А - юг,Средец,89126.0,1173.0,90299,72239,2408,2064,1806,260692,9237,7917,6928,d4c6c1e7-0f20-40f4-b1f2-b2740eadb6ec,2.606574e+05,"MULTIPOLYGON (((23.32353 42.69742, 23.32416 42...",tsgch zona a - yug,NaN
44,248,Борисова градина - Семинарията,Лозенец,NaN,2118.0,2118,1694,56,48,42,411006,136,117,102,2185c90d-6060-4e34-b203-5c4ad22cf6b8,4.109501e+05,"MULTIPOLYGON (((23.34175 42.67747, 23.34089 42...",borisova gradina - seminariyata,gradina
46,170,с. Г. и Д. Богров поле юг,Кремиковци,NaN,190.0,190,152,5,4,4,10519070,0,0,0,58e0ea7f-f5f4-47c1-adff-4752480edfa9,1.051766e+07,"MULTIPOLYGON (((23.49715 42.70708, 23.49714 42...",g. i d. bogrov pole yug,NaN
50,67,ЦГЧ Зони Б2 1 и 2,Възраждане / Триадица,77543.0,7390.0,84933,67946,2265,1941,1699,182083,12439,10660,9331,fdb1a3cd-83cf-457c-b733-cf6cfc4a5648,1.820592e+05,"MULTIPOLYGON (((23.32023 42.69805, 23.32172 42...",tsgch zoni b2 1 i 2,NaN


In [38]:
gdf = gpd.GeoDataFrame(merge, geometry='geometry', crs=districts.crs)

gdf[gdf['location'].str.contains('Sofia, Tsentar')].drop_duplicates(subset='location_clean')

,url,ad_id,listing_type,location,price_eur,area_sqm,price_per_sqm_eur,floor,central_heating,gas,...,ppl_40kvm,area_ge,dens30_ppl_kvkm,dens35_ppl_kvkm,dens40_ppl_kvkm,id,area_kv_m,geometry,regname_clean,regname_clean_2
22075,https://imoti.info/en/properties/91976678-for-...,91976678,Studio,"Sofia, Tsentar",175 000,60,"2 916,67",5th,YES,-,...,48.0,449941.0,142.0,122.0,107.0,57f8648b-7a9c-4b52-9c12-0a393c7afa25,449880.12,"MULTIPOLYGON (((23.19482 42.67822, 23.19484 42...",meditsinski tsentar suhodol,tsentar
